In [1]:
import pandas as pd
import os
import re
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import ast

from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm
from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor

import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

In [2]:
import pymatgen

from pymatgen.io.cif import CifParser
from io import StringIO

from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer


In [117]:
test_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/test.csv")
val_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/val.csv")
train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/train.csv")

In [118]:
test_df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,xrd,xrd_peak_locations,xrd_peak_intensities,atomic_numbers,TsachID,disc_sim_xrd,lattice_system
0,0,0,65,mp-865981,-0.436368,0.0000,TmMgHg2,0.000000,"['Hg', 'Mg', 'Tm']",# generated using pymatgen\ndata_TmMgHg2\n_sym...,225,DiffractionPattern\n$2\Theta$: [21.55678075 24...,"[21.556780747105403, 24.941537504017845, 35.56...","[13.285961903680525, 14.591926184240076, 100.0...","[69, 12, 80, 80]",TmMgHgHg5.04880045.04880045.048800460.060.060.0,[ 0. 0. 0. 0. ...,4.0
1,1,1,32049,mp-1103778,-2.755559,3.5845,HoWClO4,0.025353,"['Cl', 'Ho', 'O', 'W']",# generated using pymatgen\ndata_HoWClO4\n_sym...,12,DiffractionPattern\n$2\Theta$: [13.34237988 15...,"[13.342379883238058, 15.070440246206672, 17.99...","[35.121389194513256, 23.7685446066961, 39.3095...","[67, 67, 74, 74, 17, 17, 8, 8, 8, 8, 8, 8, 8, 8]",HoHoWWClClOOOOOOOO6.34135856.34135856.97127787...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,0.0
2,2,2,27350,mp-39712,-3.299936,2.4439,NaCaTaTiO6,0.012439,"['Ca', 'Na', 'O', 'Ta', 'Ti']",# generated using pymatgen\ndata_NaCaTaTiO6\n_...,7,DiffractionPattern\n$2\Theta$: [15.96727154 19...,"[15.96727153951285, 19.67006511693623, 19.8855...","[0.004290383911553776, 66.15779547527647, 30.4...","[11, 11, 20, 20, 73, 73, 22, 22, 8, 8, 8, 8, 8...",NaNaCaCaTaTaTiTiOOOOOOOOOOOO5.5505695.4564849....,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,1.0
3,3,3,26417,mp-754553,-0.021981,0.2182,Zn3N2,0.009962,"['Zn', 'N']",# generated using pymatgen\ndata_Zn3N2\n_symme...,62,DiffractionPattern\n$2\Theta$: [14.73896984 16...,"[14.73896984079199, 16.7388611273782, 21.10078...","[0.8787558771765138, 0.11562969378599479, 0.00...","[30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 3...",ZnZnZnZnZnZnZnZnZnZnZnZnNNNNNNNN3.4052135.9000...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,5.0
4,4,4,9763,mp-20332,-0.684002,0.0000,GdMgPd,0.000000,"['Gd', 'Mg', 'Pd']",# generated using pymatgen\ndata_GdMgPd\n_symm...,189,DiffractionPattern\n$2\Theta$: [13.5715734 21...,"[13.571573399184643, 21.69700166894478, 23.618...","[10.804128898540762, 14.724233778710095, 35.22...","[64, 64, 64, 12, 12, 12, 46, 46, 46]",GdGdGdMgMgMgPdPdPd7.53384827.53384824.09600390...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,1.0


In [64]:
def lattice_system(lattice_parameters, angles, tolerance=0.01):
    """
    Check if the given lattice parameters and angles define a cubic or hexagonal lattice within a given tolerance.

    Parameters:
    - lattice_parameters: List or tuple of lattice parameters [a, b, c]
    - angles: List or tuple of angles [alpha, beta, gamma]
    - tolerance: Float representing the allowed deviation for each parameter

    Returns:
    - name of the lattice system the crystal belongs to 
    """

    # Unpack lattice parametersa and angles
    a, b, c = lattice_parameters
    alpha, beta, gamma = angles
    
    ab_equal = abs(a - b) <= tolerance
    ac_equal = abs(a - c) <= tolerance
    bc_equal = abs(b - c) <= tolerance
    
    alpha_beta_equal = abs(alpha - beta) <= tolerance
    alpha_gamma_equal = abs(alpha - gamma) <= tolerance
    beta_gamma_equal = abs(beta - gamma) <= tolerance
    
    angles_90 = [abs(alpha - 90) <= tolerance, abs(beta - 90) <= tolerance, abs(gamma - 90) <= tolerance]
    angle_120 = [abs(alpha - 120) <= tolerance, abs(beta - 120) <= tolerance, abs(gamma - 120) <= tolerance]

    # Count how many angles are close to 90 and 120 degrees
    count_90 = angles_90.count(True)
    count_120 = angle_120.count(True)
    
    #count how many sides are the same
    pairs_total = ab_equal + ac_equal 
    if pairs_total == 0: 
        sides_equal = 0 
    elif pairs_total == 1: 
        sides_equal = 2
    else: 
        sides_equal = 3
        
    angle_pairs_total = alpha_beta_equal + alpha_gamma_equal
    
    if angle_pairs_total == 0: 
        angles_equal = 0 
    elif angle_pairs_total == 1: 
        angles_equal = 2
    else: 
        angles_equal = 3

    if (sides_equal == 3) and (count_90 == 3): 
        return('cubic')
    
    elif (sides_equal == 2) and (count_120 == 1) and (count_90 == 2): 
        return('hexagonal')
    
    elif (sides_equal == 3) and (angles_equal == 3) and (count_90 == 0): 
        return("rhombohedral")
    
    elif (sides_equal == 2) and (count_90 == 3):
        return("tetragonal")
    
    elif (sides_equal == 0) and (count_90 == 3): 
        return("orthorhombic") 
    
    elif (sides_equal == 0) and (count_90 == 2): 
        return("monoclinic")

    # If it is neither cubic nor hexagonal
    return 'triclinic'


In [65]:
def softmax_with_temperature(logits, temperature):
    return torch.softmax(logits / temperature, dim=-1)

In [66]:
def lattice_system_torch(batch_lengths, batch_angles, tolerance=0.01):
    tolerance = torch.tensor(tolerance)

    # Unpack lattice parameters and angles
    batch_lengths, batch_angles = batch_lengths[:, :3], batch_angles[:, :3]

    ab_equal = torch.le(torch.abs(batch_lengths[:, 0] - batch_lengths[:, 1]), tolerance).float()
    ac_equal = torch.le(torch.abs(batch_lengths[:, 0] - batch_lengths[:, 2]), tolerance).float()
    bc_equal = torch.le(torch.abs(batch_lengths[:, 1] - batch_lengths[:, 2]), tolerance).float()

    alpha_beta_equal = torch.le(torch.abs(batch_angles[:, 0] - batch_angles[:, 1]), tolerance).float()
    alpha_gamma_equal = torch.le(torch.abs(batch_angles[:, 0] - batch_angles[:, 2]), tolerance).float()
    beta_gamma_equal = torch.le(torch.abs(batch_angles[:, 1] - batch_angles[:, 2]), tolerance).float()

    angles_90 = torch.le(torch.abs(batch_angles - 90), tolerance).float()
    angles_120 = torch.le(torch.abs(batch_angles - 120), tolerance).float()

    # Convert boolean results to float for further operations
    count_90 = torch.sum(angles_90, dim=1)
    count_120 = torch.sum(angles_120, dim=1)
    
    # Differentiable approximation of side and angle equality
    pairs_total = ab_equal.float() + ac_equal.float() 
    sides_equal = torch.clamp(pairs_total, 0, 3)
    
    angle_pairs_total = alpha_beta_equal.float() + alpha_gamma_equal.float()
    angles_equal = torch.clamp(angle_pairs_total, 0, 3)
        
    # Replace return statements with differentiable computations
    # Example: using softmax to return probabilities instead of discrete class numbers
    # Adjust this part according to the specific needs of your application

    # Define scores for each lattice type
    scores = torch.zeros(batch_lengths.shape[0], 7)
    
    scores[:, 6] = torch.where((sides_equal == 2) & (count_90 == 3), 1.0, 0.0)
    scores[:, 5] = torch.where((sides_equal == 1) & (count_120 == 1) & (count_90 == 2), 1.0, 0.0)
    scores[:, 4] = torch.where((sides_equal == 2) & (angles_equal == 2) & (count_90 == 0), 1.0, 0.0)
    scores[:, 3] = torch.where((sides_equal == 1) & (count_90 == 3), 1.0, 0.0)
    scores[:, 2] = torch.where((sides_equal == 0) & (count_90 == 3), 1.0, 0.0)
    scores[:, 1] = torch.where((sides_equal == 0) & (count_90 == 2), 1.0, 0.0)
    
    scores[:, 0] = torch.where((sides_equal < 2) & (count_90 == 0), 1.0, 0.0)
    
    indices = torch.arange(scores.shape[1], dtype=torch.float32).expand_as(scores)

    weighted_indices = torch.sum(scores * indices, dim=1)
    
    # Return softmax of scores for a probability distribution over lattice types
    #return softmax_with_temperature(scores, 0.2)
    return weighted_indices

In [67]:
lengths_torch = torch.tensor([[1.0, 1.0, 5.0]], dtype=torch.float32, requires_grad=True)
angles_torch = torch.tensor([[90.0, 90.0, 120.0]], dtype=torch.float32, requires_grad=True)
lattice_system_torch(lengths_torch, angles_torch, 0.01)

tensor([5.])

In [71]:
def test_lattice_system():
    test_cases = [
        # (lattice_parameters, angles, tolerance, expected_result)
        ([1, 1, 1], [90, 90, 90], 0.01, 'cubic'),
        ([1, 1, 1.5], [90, 90, 120], 0.01, 'hexagonal'),
        ([1, 1, 1], [70, 70, 70], 0.01, 'rhombohedral'),
        ([1, 1, 1.5], [90, 90, 90], 0.01, 'tetragonal'),
        ([1, 1.2, 1.5], [90, 90, 90], 0.01, 'orthorhombic'),
        ([1, 1.2, 1.5], [90, 80, 70], 0.01, 'triclinic'),
        ([1, 1, 1], [90, 90, 90.01], 0.1, 'cubic'),
        ([1, 1, 1.02], [90, 90, 119.99], 0.02, 'hexagonal'),
        ([1, 1, 1], [60, 60, 60], 0.01, 'rhombohedral'),
        ([1, 1.02, 1.02], [90, 90, 90], 0.01, 'orthorhombic'),
        ([1, 1, 1.5], [90, 90, 121], 0.01, 'triclinic')
    ]

    for idx, (lattice_parameters, angles, tolerance, expected) in enumerate(test_cases, start=1):
        result = lattice_system(lattice_parameters, angles, tolerance)
        if result != expected:
            print(f"Test case {idx} failed: Expected {expected}, got {result}.")

# Run test cases
test_lattice_system()

In [69]:
list_of_systems = ['cubic', 'hexagonal', 'rhombohedral', 'tetragonal', 'orthorhombic', 'monoclinic', 'triclinic']

In [72]:
def test_lattice_system_torch():
    test_cases = [
        # (lattice_parameters, angles, tolerance, expected_result)
        ([1, 1, 1], [90, 90, 90], 0.01, 'cubic'),
        ([1, 1, 1.5], [90, 90, 120], 0.01, 'hexagonal'),
        ([1, 1, 1], [70, 70, 70], 0.01, 'rhombohedral'),
        ([1, 1, 1.5], [90, 90, 90], 0.01, 'tetragonal'),
        ([1, 1.2, 1.5], [90, 90, 90], 0.01, 'orthorhombic'),
        ([1, 1.2, 1.5], [90, 80, 70], 0.01, 'triclinic'),
        ([1, 1, 1], [90, 90, 90.01], 0.1, 'cubic'),
        ([1, 1, 1.02], [90, 90, 119.99], 0.02, 'hexagonal'),
        ([1, 1, 1], [60, 60, 60], 0.01, 'rhombohedral'),
        ([1, 1.02, 1.02], [90, 90, 90], 0.01, 'orthorhombic'),
        ([1, 1, 1.5], [90, 90, 121], 0.01, 'triclinic')
    ]

    for idx, (lattice_parameters, angles, tolerance, expected) in enumerate(test_cases, start=1):
        
        max_index = lattice_system_torch(lattice.lengths, lattice.angles, 0.01)

        result = list_of_systems[len(list_of_systems)-max_index-1]
        
        if result != expected:
            print(f"Test case {idx} failed: Expected {expected}, got {result}.")

# Run test cases
test_lattice_system()

In [82]:
lengths_tensor = []
angles_tensor = []

results_list = []
for index in np.arange(len(val_df)):
    cif_file = val_df['cif'][index]

    # Assume `cif_data` is a string containing the contents of the CIF file.
    cif_data = cif_file

    # Use StringIO to create a file-like object from the string.
    cif_file_like = StringIO(cif_data)

    # Create a CIF parser with this file-like object.
    parser = CifParser(cif_file_like)

    # This will return a Structure object.
    structure = parser.get_structures()[0]

    # Create a SpacegroupAnalyzer object
    spacegroup_analyzer = SpacegroupAnalyzer(structure)

    # Get the lattice system
    lattice_sys = spacegroup_analyzer.get_lattice_type()

    # Access the lattice object from the structure
    lattice = structure.lattice
    
    result = lattice_system(lattice.lengths, lattice.angles, 0.01)
    results_list.append(result)
    lengths_tensor.append(lattice.lengths)
    angles_tensor.append(lattice.angles)    

lengths_tensor = torch.tensor(lengths_tensor, dtype=torch.float32, requires_grad=True)
angles_tensor = torch.tensor(angles_tensor, dtype=torch.float32, requires_grad=True)

tensor_results = lattice_system_torch(lengths_tensor, angles_tensor, 0.01)

In [83]:
tensor_results

tensor([4., 0., 1., 5., 1., 4., 5., 0., 6., 0., 0., 2., 0., 0., 4., 2., 4., 1.,
        0., 4., 2., 5., 0., 5., 0., 4., 4., 0., 0., 3., 1., 4., 1., 4., 6., 0.,
        3., 2., 4., 4., 4., 2., 4., 2., 6., 4., 4., 1., 0., 1., 1., 2., 1., 0.,
        0., 1., 2., 4., 4., 1., 0., 5., 1., 4., 6., 2., 2., 4., 4., 1., 0., 4.,
        1., 0., 2., 0., 4., 0., 4., 0., 4., 2., 5., 3., 0., 2., 0., 1., 1., 0.,
        4., 6., 0., 0., 0., 4., 2., 0., 0., 0., 4., 4., 3., 0., 0., 0., 1., 4.,
        0., 2., 1., 0., 1., 6., 0., 1., 4., 5., 1., 2., 0., 1., 3., 2., 0., 2.,
        2., 3., 6., 2., 5., 0., 2., 0., 4., 2., 1., 0., 4., 5., 0., 1., 5., 4.,
        0., 0., 0., 0., 0., 4., 0., 0., 3., 0., 0., 4., 4., 1., 1., 0., 1., 0.,
        0., 1., 4., 4., 4., 2., 4., 0., 4., 4., 0., 1., 4., 0., 0., 0., 1., 1.,
        0., 4., 1., 0., 0., 2., 0., 1., 1., 0., 4., 6., 4., 4., 0., 5., 1., 6.,
        0., 6., 4., 4., 2., 0., 4., 0., 5., 6., 4., 1., 1., 0., 0., 5., 4., 5.,
        0., 1., 4., 0., 4., 4., 2., 0., 

In [84]:
for index in np.arange(256): 
    max_index = int(tensor_results[index])

    result = list_of_systems[len(list_of_systems)-max_index-1]
    if not (result == results_list[index]): 
        print(index)
        print(result)
        print(results_list[index])

In [85]:
val_df['lattice_system'] = tensor_results

In [86]:
lengths_tensor = []
angles_tensor = []

results_list = []
for index in np.arange(len(val_df)):
    cif_file = val_df['cif'][index]

    # Assume `cif_data` is a string containing the contents of the CIF file.
    cif_data = cif_file

    # Use StringIO to create a file-like object from the string.
    cif_file_like = StringIO(cif_data)

    # Create a CIF parser with this file-like object.
    parser = CifParser(cif_file_like)

    # This will return a Structure object.
    structure = parser.get_structures()[0]

    # Create a SpacegroupAnalyzer object
    spacegroup_analyzer = SpacegroupAnalyzer(structure)

    # Get the lattice system
    lattice_sys = spacegroup_analyzer.get_lattice_type()

    # Access the lattice object from the structure
    lattice = structure.lattice
    
    result = lattice_system(lattice.lengths, lattice.angles, 0.01)
    results_list.append(result)
    lengths_tensor.append(lattice.lengths)
    angles_tensor.append(lattice.angles)    


0       4.0
1       0.0
2       1.0
3       5.0
4       1.0
       ... 
9042    5.0
9043    4.0
9044    0.0
9045    0.0
9046    4.0
Name: lattice_system, Length: 9047, dtype: float32

In [88]:
lengths_tensor = []
angles_tensor = []

results_list = []
for index in np.arange(len(test_df)):
    cif_file = test_df['cif'][index]

    # Assume `cif_data` is a string containing the contents of the CIF file.
    cif_data = cif_file

    # Use StringIO to create a file-like object from the string.
    cif_file_like = StringIO(cif_data)

    # Create a CIF parser with this file-like object.
    parser = CifParser(cif_file_like)

    # This will return a Structure object.
    structure = parser.get_structures()[0]

    # Create a SpacegroupAnalyzer object
    spacegroup_analyzer = SpacegroupAnalyzer(structure)

    # Get the lattice system
    lattice_sys = spacegroup_analyzer.get_lattice_type()

    # Access the lattice object from the structure
    lattice = structure.lattice
    
    result = lattice_system(lattice.lengths, lattice.angles, 0.01)
    results_list.append(result)
    lengths_tensor.append(lattice.lengths)
    angles_tensor.append(lattice.angles)    

lengths_tensor = torch.tensor(lengths_tensor, dtype=torch.float32, requires_grad=True)
angles_tensor = torch.tensor(angles_tensor, dtype=torch.float32, requires_grad=True)

tensor_results = lattice_system_torch(lengths_tensor, angles_tensor, 0.01)

In [89]:
tensor_results

tensor([4., 0., 1., 5., 1., 4., 5., 0., 6., 0., 0., 2., 0., 0., 4., 2., 4., 1.,
        0., 4., 2., 5., 0., 5., 0., 4., 4., 0., 0., 3., 1., 4., 1., 4., 6., 0.,
        3., 2., 4., 4., 4., 2., 4., 2., 6., 4., 4., 1., 0., 1., 1., 2., 1., 0.,
        0., 1., 2., 4., 4., 1., 0., 5., 1., 4., 6., 2., 2., 4., 4., 1., 0., 4.,
        1., 0., 2., 0., 4., 0., 4., 0., 4., 2., 5., 3., 0., 2., 0., 1., 1., 0.,
        4., 6., 0., 0., 0., 4., 2., 0., 0., 0., 4., 4., 3., 0., 0., 0., 1., 4.,
        0., 2., 1., 0., 1., 6., 0., 1., 4., 5., 1., 2., 0., 1., 3., 2., 0., 2.,
        2., 3., 6., 2., 5., 0., 2., 0., 4., 2., 1., 0., 4., 5., 0., 1., 5., 4.,
        0., 0., 0., 0., 0., 4., 0., 0., 3., 0., 0., 4., 4., 1., 1., 0., 1., 0.,
        0., 1., 4., 4., 4., 2., 4., 0., 4., 4., 0., 1., 4., 0., 0., 0., 1., 1.,
        0., 4., 1., 0., 0., 2., 0., 1., 1., 0., 4., 6., 4., 4., 0., 5., 1., 6.,
        0., 6., 4., 4., 2., 0., 4., 0., 5., 6., 4., 1., 1., 0., 0., 5., 4., 5.,
        0., 1., 4., 0., 4., 4., 2., 0., 

In [90]:
test_df['lattice_system'] = tensor_results

In [91]:
lengths_tensor = []
angles_tensor = []

results_list = []
for index in np.arange(len(train_df)):
    cif_file = train_df['cif'][index]

    # Assume `cif_data` is a string containing the contents of the CIF file.
    cif_data = cif_file

    # Use StringIO to create a file-like object from the string.
    cif_file_like = StringIO(cif_data)

    # Create a CIF parser with this file-like object.
    parser = CifParser(cif_file_like)

    # This will return a Structure object.
    structure = parser.get_structures()[0]

    # Create a SpacegroupAnalyzer object
    spacegroup_analyzer = SpacegroupAnalyzer(structure)

    # Get the lattice system
    lattice_sys = spacegroup_analyzer.get_lattice_type()

    # Access the lattice object from the structure
    lattice = structure.lattice
    
    result = lattice_system(lattice.lengths, lattice.angles, 0.01)
    results_list.append(result)
    lengths_tensor.append(lattice.lengths)
    angles_tensor.append(lattice.angles)    

lengths_tensor = torch.tensor(lengths_tensor, dtype=torch.float32, requires_grad=True)
angles_tensor = torch.tensor(angles_tensor, dtype=torch.float32, requires_grad=True)

tensor_results = lattice_system_torch(lengths_tensor, angles_tensor, 0.01)

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


In [92]:
tensor_results

tensor([0., 0., 4., 2., 0., 0., 1., 0., 1., 1., 0., 2., 5., 5., 4., 0., 5., 0.,
        4., 3., 1., 4., 1., 1., 5., 1., 4., 1., 0., 4., 5., 0., 6., 4., 0., 5.,
        1., 6., 4., 0., 4., 0., 0., 5., 5., 0., 0., 0., 0., 0., 0., 4., 5., 0.,
        0., 1., 0., 0., 0., 1., 5., 0., 0., 0., 1., 1., 0., 5., 0., 1., 5., 2.,
        2., 4., 3., 5., 4., 0., 0., 4., 0., 2., 1., 2., 0., 0., 4., 2., 4., 4.,
        1., 0., 0., 0., 4., 0., 3., 0., 4., 4., 0., 4., 5., 1., 2., 5., 0., 0.,
        1., 2., 0., 1., 4., 0., 2., 0., 3., 0., 3., 1., 1., 0., 0., 1., 1., 0.,
        0., 1., 4., 4., 6., 1., 0., 0., 4., 1., 4., 1., 1., 5., 0., 0., 4., 3.,
        4., 0., 0., 0., 0., 5., 2., 1., 0., 4., 2., 0., 1., 1., 6., 4., 0., 4.,
        4., 0., 0., 2., 4., 0., 3., 0., 4., 4., 4., 5., 1., 5., 5., 4., 2., 4.,
        2., 3., 1., 0., 3., 6., 0., 4., 0., 4., 2., 1., 0., 4., 1., 0., 6., 0.,
        0., 4., 0., 0., 4., 3., 1., 0., 4., 0., 2., 0., 2., 4., 1., 0., 4., 6.,
        1., 0., 1., 0., 1., 5., 1., 0., 

In [93]:
train_df['lattice_system'] = tensor_results

In [103]:
import datetime

In [97]:
pairings = {"/home/gridsan/tmackey/cdvae/data/mp_20/test.csv":test_df, 
            "/home/gridsan/tmackey/cdvae/data/mp_20/val.csv":val_df,
            "/home/gridsan/tmackey/cdvae/data/mp_20/train.csv":train_df}

In [99]:
value = "/home/gridsan/tmackey/cdvae/data/mp_20/test.csv"

In [107]:
current_date = datetime.date.today()
print("Current date:", current_date)


Current date: 2023-11-10


In [115]:
def adjust_datasets(pairings):
    current_date = str(datetime.date.today())

    for original_filename, new_data in pairings.items():
        new_filename = original_filename[:-4] + current_date + ".csv"
        print(new_filename)
        print(original_filename)
        !mv $original_filename $new_filename
        new_data.to_csv(original_filename)
    

In [116]:
adjust_datasets(pairings)

/home/gridsan/tmackey/cdvae/data/mp_20/test2023-11-10.csv
/home/gridsan/tmackey/cdvae/data/mp_20/test.csv
/home/gridsan/tmackey/cdvae/data/mp_20/val2023-11-10.csv
/home/gridsan/tmackey/cdvae/data/mp_20/val.csv
/home/gridsan/tmackey/cdvae/data/mp_20/train2023-11-10.csv
/home/gridsan/tmackey/cdvae/data/mp_20/train.csv


In [ ]:
train_df.to_csv()